In [6]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np

project_root = os.path.abspath('/home/user/perso/trading/alphalab')

if project_root not in sys.path:
    sys.path.append(project_root)


### Exercice 1 : Émancipation OLS à passe unique et correction du biais

Un robot de trading souhaite calibrer de manière autonome la droite d'équilibre entre un nouvel indice sectoriel (Y) et son actif de référence (X). Le processeur réalise un balayage en une seule passe sur un échantillon historique réduit à $T = 4$ observations et stocke les indicateurs scalaires cumulés suivants :
*   $\sum_{t=1}^{4} x_t = 10.0$
*   $\sum_{t=1}^{4} y_t = 20.0$
*   $\sum_{t=1}^{4} x_t^2 = 30.0$
*   $\sum_{t=1}^{4} y_t^2 = 114.0$
*   $\sum_{t=1}^{4} x_t y_t = 55.0$

1. **Calcul des coefficients :**
   * a. Calculer la valeur numérique du dénominateur commun.
   * b. Calculer la valeur exacte de la pente optimale $\hat{b}$ de l'estimateur OLS.
   * c. Calculer la valeur exacte de l'intercepte optimal $\hat{a}$ en utilisant la formule brute développée.
   * d. Écrire l'équation définitive de la droite d'équilibre de ce modèle.

2. **Mesure de la volatilité du spread :**
   * a. Calculer la valeur numérique de la Somme des Carrés des Résidus (RSS) en utilisant la formule rapide à passe unique démontrée à la Section 9.
   * b. Déterminer le nombre exact de degrés de liberté du système et justifier la valeur du diviseur.
   * c. En déduire la valeur numérique exacte de l'écart-type sans biais du spread ($\sigma_e$).

In [7]:
T = 4
x_sum = 10.0
y_sum = 20.0
x_mean = (x_sum / T)
y_mean = (y_sum / T)
x2_sum = 30.0
y2_sum = 114.0
xy_sum = 55.0

# 1. Coefs
common_denom = (T * x2_sum) - (x_sum ** 2)

coef = ((T * xy_sum) - (x_sum * y_sum)) / common_denom
intercept = ((y_sum * x2_sum) - (x_sum * xy_sum)) / common_denom
fit = lambda x: intercept + coef * x

print(f'intercept: {intercept}')
print(f'coef: {coef}')

# 2. Spread
rss = (y2_sum - T * (y_mean ** 2)) - coef * (xy_sum - T * x_mean * y_mean)
std = np.sqrt([(rss / (T - 2))])

print(f'rss: {rss}')
print(f'std: {std[0]:.4f}')

intercept: 2.5
coef: 1.0
rss: 9.0
std: 2.1213



### Exercice 2 : Logique opérationnelle du Hedge Ratio et neutralité

Le robot a calibré la relation statistique entre l'EUR/USD (Y) et le GBP/USD (X) sur une fenêtre glissante. Le moteur d'optimisation extrait une pente optimale $\hat{b}_t = 1.25$.

À cet instant précis, le spread subit une anomalie géométrique majeure et franchit la borne supérieure du modèle, générant un Z-Score de $Z_t = +2.50$.

1. **Génération du signal :** Déterminer le signal opérationnel que le robot doit déclencher (Achat, Vente ou Neutralité du spread).
2. **Gestion des jambes de trading :** En déduire la nature exacte des positions à ouvrir simultanément sur les deux parités du Forex (Long ou Short pour l'EUR/USD, Long ou Short pour le GBP/USD).
3. **Calcul de la taille des positions :** Si le robot décide d'engager un volume standard de 1.0 lot sur la jambe EUR/USD, calculer le volume exact de lots à ouvrir sur la jambe GBP/USD pour garantir la neutralité directionnelle face au Dollar.
4. **Scénario comptable :** Si le Dollar américain (USD) subit une hausse globale, provoquant l'effondrement simultané des deux paires, et que l'EUR/USD chute de 80 pips tandis que le GBP/USD chute de 120 pips, calculer le bilan comptable net en pips pour le robot lors du retour à l'équilibre ($Z = 0$).

In [23]:
# Signal
coef = 1.25
sell = 1.0
z_score = 2.50
buy = sell * coef
print(f'SELL {sell:.3f} EUR/USD, BUY {buy:.3f} GBP/USD')

# Gain
eurusd_pips = -80.0 * (-sell)
gbpusd_pips = -120.0 * buy
net_pips = eurusd_pips + gbpusd_pips
print(f'eur/usd gain: {eurusd_pips} pips')
print(f'gbp/usd gain: {gbpusd_pips} pips')
print(f'net: {net_pips:.1f} pips')

SELL 1.000 EUR/USD, BUY 1.250 GBP/USD
eur/usd gain: 80.0 pips
gbp/usd gain: -150.0 pips
net: -70.0 pips


# Atelier

In [29]:
from enl.bivariate_model import BivariateModel

# Instantiating the new OLS engine
model = BivariateModel()

# Injecting Section 7 numerical data (T = 3)
x_market = np.array([1.0, 2.0, 3.0])
y_market = np.array([2.0, 4.0, 5.0])

# Executing calibration pipeline
model.fit(x_market, y_market)

print("--- OLS CALIBRATION DIAGNOSTICS ---")
print(f"Intercept (a_hat) : {model.intercept_:.4f}  | Expected: 0.6667")
print(f"Slope (b_hat)     : {model.coef_:.4f}  | Expected: 1.5000")
print(f"Volatility (sig)  : {model.sigma_e_:.4f}  | Expected: 0.4082")

--- OLS CALIBRATION DIAGNOSTICS ---
Intercept (a_hat) : 0.6667  | Expected: 0.6667
Slope (b_hat)     : 1.5000  | Expected: 1.5000
Volatility (sig)  : 0.4082  | Expected: 2.1213
